In [10]:

import pandas as pd
import numpy as np
import spacy
import textstat
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bert_score import score
from rouge_score import rouge_scorer
import nltk
from nltk.util import ngrams
import os
from openai import OpenAI

nltk.download('punkt')

nlp = spacy.load("en_core_web_sm")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


class ContextAwareEvaluator:

    def __init__(self, query, output):
        self.query = query
        self.output = output

    ########################################
    # Semantic Similarity
    ########################################

    def semantic_similarity(self, query, response):
        q_emb = embedding_model.encode([query])
        r_emb = embedding_model.encode([response])

        sim = cosine_similarity(q_emb, r_emb)[0][0]
        return float(sim)

    ########################################
    # BERTScore
    ########################################

    def bertscore(self, query, response):
        P, R, F1 = score([response], [query], lang='en', verbose=False)
        return float(F1[0])

    ########################################
    # ROUGE
    ########################################

    def rouge_score_eval(self, query, response):
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
        scores = scorer.score(query, response)

        return {
            'rouge1': scores['rouge1'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        }

    ########################################
    # Entity Overlap
    ########################################

    def entity_overlap(self, query, response):
        q_doc = nlp(query)
        r_doc = nlp(response)

        q_entities = set([ent.text.lower() for ent in q_doc.ents])
        r_entities = set([ent.text.lower() for ent in r_doc.ents])

        if len(q_entities) == 0:
            return 0

        overlap = len(q_entities.intersection(r_entities)) / len(q_entities)
        return overlap

    ########################################
    # Keyword Coverage
    ########################################

    def keyword_coverage(self, query, response):
        vectorizer = TfidfVectorizer(stop_words='english')

        tfidf = vectorizer.fit_transform([query, response])

        query_words = set(query.lower().split())
        response_words = set(response.lower().split())

        if len(query_words) == 0:
            return 0

        return len(query_words.intersection(response_words)) / len(query_words)

    ########################################
    # Specificity Score
    ########################################

    def specificity_score(self, response):
        doc = nlp(response)

        named_entities = len(doc.ents)
        noun_chunks = len(list(doc.noun_chunks))
        unique_words = len(set(response.split()))
        total_words = len(response.split())

        if total_words == 0:
            return 0

        specificity = (
            (named_entities * 0.4) +
            (noun_chunks * 0.3) +
            ((unique_words / total_words) * 0.3)
        )

        return specificity

    ########################################
    # Diversity Metrics
    ########################################

    def distinct_n(self, text, n=2):
        tokens = nltk.word_tokenize(text.lower())

        if len(tokens) < n:
            return 0

        ng = list(ngrams(tokens, n))

        return len(set(ng)) / len(ng)

    ########################################
    # Readability
    ########################################

    def readability(self, text):
        return textstat.flesch_reading_ease(text)

    ########################################
    # Final Score
    ########################################

    def final_score(self, metrics):
        return (
            0.30 * metrics['semantic_similarity'] +
            0.20 * metrics['bertscore'] +
            0.15 * metrics['entity_overlap'] +
            0.15 * metrics['specificity'] +
            0.10 * metrics['distinct_2'] +
            0.10 * metrics['readability_normalized']
        )

    ########################################
    # Evaluate Single Response
    ########################################

    def evaluate_response(self, query, response):

        semantic_sim = self.semantic_similarity(query, response)
        bert_f1 = self.bertscore(query, response)

        rouge = self.rouge_score_eval(query, response)

        entity_overlap = self.entity_overlap(query, response)
        keyword_cov = self.keyword_coverage(query, response)

        specificity = self.specificity_score(response)

        distinct_1 = self.distinct_n(response, 1)
        distinct_2 = self.distinct_n(response, 2)

        readability = self.readability(response)
        readability_normalized = min(max(readability / 100, 0), 1)

        metrics = {
            'semantic_similarity': semantic_sim,
            'bertscore': bert_f1,
            'rouge1': rouge['rouge1'],
            'rougeL': rouge['rougeL'],
            'entity_overlap': entity_overlap,
            'keyword_coverage': keyword_cov,
            'specificity': specificity,
            'distinct_1': distinct_1,
            'distinct_2': distinct_2,
            'readability': readability,
            'readability_normalized': readability_normalized
        }

        metrics['final_score'] = self.final_score(metrics)

        return metrics

    ########################################
    # Full Dataset Evaluation
    ########################################

    def run(self):

        metrics = self.evaluate_response(self.query, self.output)
        
        return metrics


if __name__ == '__main__':

    query = 'What is machine learning?'
    output = '''**machine learning (ml)** is a subset of artificial intelligence (ai) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed.

### key characteristics:

* **data-driven**: ml algorithms learn from data, which can come in various forms, such as images, text, audio, or sensor readings.
* **model-based**: ml algorithms create models that represent the relationships between the data and the desired output.
* **self-improving**: ml algorithms can improve their performance over time by adjusting their models based on new data.

### types of machine learning:

1. **supervised learning**: the algorithm learns from labeled data to make predictions on new, unseen data. examples: image classification, sentiment analysis.
2. **unsupervised learning**: the algorithm discovers patterns or relationships in unlabeled data. examples: clustering, dimensionality reduction.
3. **reinforcement learning**: the algorithm learns by interacting with an environment and receiving feedback in the form of rewards or penalties. examples: game playing, robotics.

### machine learning workflow:

1. **data collection**: gather relevant data from various sources.
2. **data preprocessing**: clean, transform, and prepare the data for modeling.
3. **model selection**: choose a suitable ml algorithm and configure its hyperparameters.
4. **model training**: train the model on the prepared data.
5. **model evaluation**: assess the model's performance on a test dataset.
6. **model deployment**: deploy the trained model in a production-ready environment.

### applications of machine learning:

* **computer vision**: image recognition, object detection, segmentation.
* **natural language processing**: text classification, language translation, sentiment analysis.
* **predictive maintenance**: predicting equipment failures, scheduling maintenance.
* **recommendation systems**: personalized product recommendations.

### real-world examples:

* virtual assistants like siri, alexa, and google assistant use ml to understand voice commands.
* self-driving cars use ml to detect objects, predict movements, and make decisions.
* online advertising platforms use ml to personalize ads based on user behavior.'''

    evaluator = ContextAwareEvaluator(query, output)

    results = evaluator.run()

    print("=" * 80)
    print("EVALUATION RESULTS")
    print("=" * 80)

    print(results)
    

[nltk_data] Downloading package punkt to /Users/pranitdas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EVALUATION RESULTS
{'semantic_similarity': 0.7348973751068115, 'bertscore': 0.7865030765533447, 'rouge1': 0.020134228187919462, 'rougeL': 0.020134228187919462, 'entity_overlap': 0, 'keyword_coverage': 0.5, 'specificity': 38.59072847682119, 'distinct_1': 0.3969631236442516, 'distinct_2': 0.6978260869565217, 'readability': 19.792556097561004, 'readability_normalized': 0.19792556097561004, 'final_score': 6.2559542641591035}


In [7]:
# Groq
api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


In [ ]:
model1, model2 = "openai/gpt-oss-120b", "meta-llama/llama-4-scout-17b-16e-instruct"
query = 'what is machine learning?'
chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"""{query}""",
        }
    ],
    model="meta-llama/llama-4-scout-17b-16e-instruct",
)

reply = chat_completion.choices[0].message.content.lower()

print(reply)

**machine learning (ml)** is a subset of artificial intelligence (ai) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed.

### key characteristics:

* **data-driven**: ml algorithms learn from data, which can come in various forms, such as images, text, audio, or sensor readings.
* **model-based**: ml algorithms create models that represent the relationships between the data and the desired output.
* **self-improving**: ml algorithms can improve their performance over time by adjusting their models based on new data.

### types of machine learning:

1. **supervised learning**: the algorithm learns from labeled data to make predictions on new, unseen data. examples: image classification, sentiment analysis.
2. **unsupervised learning**: the algorithm discovers patterns or relationships in unlabeled data. examples: clustering, dimensionality reduction.
3. **reinforcement learning**: the algorithm learns by interactin